In [2]:
!pip install -Uqqq sentence-transformers faiss-cpu python-telegram-bot PyPDF2 tqdm requests yandexcloud

In [4]:
pip install --upgrade PyPDF2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip freeze > requirements.txt

In [4]:
import os
import PyPDF2
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

DATA_DIR = "sp_data"
INDEX_PATH = "sp_index.faiss"
METADATA_PATH = "metadata.json"

c:\Users\uryup\OneDrive\Documents\GitHub\Portfolio\sp_docs_llm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
os.makedirs(DATA_DIR, exist_ok=True)
print(faiss.__version__)

1.11.0


In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Извлекает текст из PDF с сохранением структуры"""
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

In [ ]:
def clean_text(text):
    """Очистка и нормализация текста"""
    text = re.sub(r'\s+', ' ', text)  # Удаление лишних пробелов
    text = re.sub(r'[^\w\s.,;:!?()\-–%№«»§]', '', text)  # Удаление спецсимволов
    text = text.lower()

    return text.strip()

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

In [ ]:
# Обработка документов
documents = []
embeddings = []

for file in tqdm(os.listdir(DATA_DIR), desc="Обработка документов"):
    if file.endswith(".pdf"):
        file_path = os.path.join(DATA_DIR, file)
        text = extract_text_from_pdf(file_path)
        cleaned_text = clean_text(text)
        chunks = chunk_text(cleaned_text)

        for i, chunk in enumerate(chunks):
            # Сохраняем метаданные
            documents.append({
                "doc_id": f"{file}_{i}",
                "source": file,
                "chunk_index": i,
                "text": chunk
            })

            # Создаем эмбеддинг
            embedding = model.encode([chunk])[0]
            embeddings.append(embedding)

Обработка документов:   3%|▎         | 2/61 [00:00<00:06,  8.63it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Обработка документов: 100%|██████████| 61/61 [03:21<00:00,  3.31s/it]


In [ ]:
# Создание индекса FAISS
embeddings = np.array(embeddings).astype('float32')
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [ ]:
# Сохранение индекса и метаданных
faiss.write_index(index, INDEX_PATH)
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    import json
    json.dump(documents, f, ensure_ascii=False)

print(f"✅ База знаний создана: {len(documents)} чанков")

✅ База знаний создана: 433 чанков
